# 02 – Huấn luyện CTGAN (AI1) và sinh dữ liệu (AI3)
Chạy sau notebook 01. Luồng: chuẩn bị dữ liệu → huấn luyện CTGAN → sinh dữ liệu theo quota từng lớp → kiểm tra nhanh.

**Lần đầu hãy để `SMOKE_TEST = True`** (chạy vài phút để chắc code chạy). Sau đó đặt `False` để huấn luyện thật.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, data

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

DATASET = "unsw_nb15"   # đổi thành "cicids2017" để chạy dataset kia

from src.ctgan_pipeline import train_ctgan, sample_by_class, quick_fidelity

SMOKE_TEST = False   # True để chạy thử nhanh, False để chạy đầy đủ
EPOCHS = 5 if SMOKE_TEST else 300
CAP_PER_CLASS = 500 if SMOKE_TEST else 3000   # giới hạn số dòng mỗi lớp lớn (lớp hiếm giữ nguyên)


## 1. Chuẩn bị dữ liệu cho CTGAN

In [2]:
train, test = data.load(DATASET)
real, LABEL, DISCRETE = data.prepare_for_ctgan(train, DATASET, cap_per_class=CAP_PER_CLASS)
print("Bảng huấn luyện:", real.shape)
print("Cột rời rạc:", DISCRETE)
display(data.class_counts(real, LABEL))

Bảng huấn luyện: (23009, 53)
Cột rời rạc: ['attack_cat', 'proto', 'service', 'state', 'sttl', 'dttl', 'swin', 'dwin', 'trans_depth', 'ct_state_ttl', 'is_ftp_login', 'ct_ftp_cmd', 'is_sm_ips_ports']


,attack_cat,rows,percent,rare
0,Exploits,3000,13.038,False
1,DoS,3000,13.038,False
2,Normal,3000,13.038,False
3,Reconnaissance,3000,13.038,False
4,Fuzzers,3000,13.038,False
5,Generic,3000,13.038,False
6,Analysis,2000,8.692,False
7,Backdoor,1746,7.588,False
8,Shellcode,1133,4.924,False
9,Worms,130,0.565,True


In [3]:
from ctgan import CTGAN
model = CTGAN.load(str(config.OUT_DIR / f"ctgan_{DATASET}.pt"))

## 2. Huấn luyện CTGAN

In [4]:
import time
t0 = time.time()
model = train_ctgan(real, DISCRETE, epochs=EPOCHS, batch_size=500, pac=10, cuda=False)
print(f"Huấn luyện xong sau {time.time()-t0:.0f} giây")

config.OUT_DIR.mkdir(exist_ok=True)
model.save(str(config.OUT_DIR / f"ctgan_{DATASET}.pt"))

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Gen. (-00.70) | Discrim. (+00.52): 100%|██████████| 300/300 [1:12:20<00:00, 14.47s/it]


Huấn luyện xong sau 4465 giây


## 3. Sinh dữ liệu theo quota từng lớp (AI3)
Sinh số bản ghi **cố định cho mỗi lớp**, không theo tỷ lệ gốc, để lớp hiếm không bị lấn át.

In [5]:
classes = real[LABEL].unique()
QUOTA = 200 if SMOKE_TEST else 2000
quotas = {c: QUOTA for c in classes}
synth = sample_by_class(model, LABEL, quotas)
print(synth.shape)
synth.to_csv(config.OUT_DIR / f"synthetic_ctgan_{DATASET}.csv", index=False)
synth.head(20)

         class  quota  got  drawn  match_rate
      Analysis   2000 2000  47794    0.041846
      Backdoor   2000 2000  56010    0.035708
           DoS   2000 2000  12234    0.163479
      Exploits   2000 2000  12285    0.162800
       Fuzzers   2000 2000  22012    0.090860
       Generic   2000 2000 106010    0.018866
        Normal   2000 2000   8946    0.223564
Reconnaissance   2000 2000  30519    0.065533
     Shellcode   2000 2000  95104    0.021030
         Worms   2000  886 456010    0.001943
CẢNH BÁO: các lớp thiếu so với quota: ['Worms']
(18886, 53)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,log_dur,log_sbytes,log_dbytes,log_sload,log_dload,log_rate,log_sjit,log_djit,log_sinpkt,log_dinpkt
0,-0.129923,udp,unspecified,INT,0,1,490,1948,114201.904743,254,0,8.614781e+07,9892.066864,-1,-1,44.921457,0.752881,455.364777,16.105736,0,-12504986,-16373873,0,0.001007,-0.000136,0.000277,48,0,0,-42,3,2,3,2,3,8,0,0,0,2,15,0,Analysis,-0.002988,5.287293,0.003782,17.864783,0.028776,12.558832,0.024458,0.008600,0.029010,0.009803
1,-0.031154,unas,unspecified,INT,1,0,497,842,245506.307673,254,0,8.755506e+08,13924.075039,0,1,54.567241,-0.515631,501.364553,-28.579828,0,-7974920,-3215694,0,0.000611,-0.000853,-0.000352,51,2,0,-239,8,2,3,2,2,5,0,0,0,2,6,0,Analysis,0.002057,4.383432,-0.035736,19.026687,-0.069685,11.614933,0.029906,0.020219,0.030092,0.025746
2,0.629374,tcp,http,FIN,5,7,1553,2332,-186.152060,254,252,2.636484e+06,10094.721885,1,6,84.899561,20.299659,1910.036747,91.280594,255,3164962696,1357277291,255,0.107471,0.056378,0.060594,80,93,1,-242,5,1,1,1,1,6,0,0,0,1,1,0,Analysis,0.777596,6.730932,5.654591,9.482651,9.192202,2.790324,7.720874,4.842898,3.853352,4.341245
3,0.536322,tcp,unspecified,CON,23,2,243,618,182.653569,254,0,4.505368e+06,7830.226686,0,1,46.116440,6.527032,-10.466083,144.789976,255,-11228689,-11575361,255,0.000989,0.116970,0.071018,91,45,0,289,4,0,4,2,3,5,0,0,0,3,13,0,Analysis,0.313529,8.537321,-0.027817,13.156279,-0.029445,3.116231,7.276671,-0.005869,4.486775,0.025447
4,-0.084196,egp,unspecified,INT,3,0,2594,1278,118428.298218,254,0,1.122221e+08,-1809.684808,-1,0,47.240591,-3.438843,567.718963,-18.553050,0,107896,-26698652,0,0.002417,0.000563,0.000441,81,0,0,110,6,2,3,3,2,6,0,0,0,16,6,0,Analysis,0.003427,5.285730,-0.000573,18.284784,0.036737,11.586406,-0.028255,-0.024383,0.028206,0.005037
5,23.593386,tcp,unspecified,CON,3,10,1689,127,864.691147,254,252,3.133388e+04,1837.693340,2,1,674.576110,1868.952945,18177.482025,73.623155,255,1839958910,-25207183,255,0.144983,0.119733,0.111118,53,48,0,148,5,1,9,3,1,4,0,0,0,4,7,0,Analysis,0.783795,5.451308,5.085759,4.255235,7.343974,-0.464897,9.266671,5.355965,7.548304,0.036634
6,-0.097361,unas,unspecified,INT,1,0,733,1447,110879.656254,254,0,1.045753e+08,-3025.285264,1,0,36.786762,-5.390831,743.349176,4.033112,0,6637156,-21352336,0,0.001328,0.000109,-0.001516,88,0,0,-452,5,2,3,3,2,6,0,0,0,5,7,0,Analysis,-0.001396,5.280754,-0.013051,18.772436,-0.033545,11.996834,0.033108,-0.001184,0.047130,0.011048
7,-0.055122,vmtp,unspecified,INT,4,0,-211,131,103597.939324,254,0,9.324760e+07,6711.728000,0,0,33.943215,5.325922,72.255973,135.529308,0,-21610910,-12216102,0,0.001749,0.000496,0.000710,94,-1,0,534,5,2,2,3,2,3,0,0,0,2,5,0,Analysis,-0.006257,5.294391,-0.029768,17.861829,-0.023128,11.590370,-0.025519,0.039622,0.036511,0.010907
8,1.311282,tcp,unspecified,FIN,31,6,2257,1293,3991.842928,62,252,1.842190e+06,12659.059450,0,5,109.443004,53.856206,12794.598734,126.633107,255,2442144562,525055266,255,0.154825,0.137876,0.103324,78,147,1,176,3,1,1,1,1,1,0,0,1,1,1,0,Analysis,0.589029,6.660292,7.024894,8.393467,8.240165,2.515079,9.539343,5.385200,4.307459,4.556682
9,-0.130354,unas,unspecified,INT,1,0,441,2319,109043.873195,254,0,9.437816e+07,-8142.386998,-1,0,81.184212,-1.447413,327.020043,49.747785,0,6544303,-17055778,0,0.000661,-0.000320,-0.000814,77,2,0,208,4,2,3,3,3,6,0,0,0,3,8,0,Analysis,0.000328,5.278120,0.003443,17.869800,0.019372,11.504834,-0.014904,0.021961,0.019660,-0.017686


## 4. Kiểm tra nhanh chất lượng
Khoảng cách giữa dữ liệu thật và dữ liệu sinh theo từng cột (càng gần 0 càng giống). Đây chỉ là kiểm tra sơ bộ; phần fidelity chính thức thuộc nhóm Data Science (DS3).

In [6]:
fid = quick_fidelity(real, synth, DISCRETE)
print("Khoảng cách trung bình:", round(fid["distance"].mean(), 3))
fid.head(15)

Khoảng cách trung bình: 0.242


,column,type,distance
0,response_body_len,numeric,0.740178
1,dtcpb,numeric,0.598539
2,tcprtt,numeric,0.520206
3,dbytes,numeric,0.517910
4,dinpkt,numeric,0.498994
5,sinpkt,numeric,0.465302
6,dmean,numeric,0.462688
7,sbytes,numeric,0.456816
8,log_dbytes,numeric,0.413640
9,djit,numeric,0.392844


In [7]:
cmp = pd.DataFrame({"thật": real[LABEL].value_counts(), "sinh": synth[LABEL].value_counts()}).fillna(0).astype(int)
cmp

,thật,sinh
attack_cat,,
Analysis,2000,2000
Backdoor,1746,2000
DoS,3000,2000
Exploits,3000,2000
Fuzzers,3000,2000
Generic,3000,2000
Normal,3000,2000
Reconnaissance,3000,2000
Shellcode,1133,2000


## NExt Step

- Tăng `EPOCHS` và `CAP_PER_CLASS` (đặt `SMOKE_TEST = False`).
- Với lớp rất hiếm (ví dụ Worms), kiểm tra dữ liệu sinh có thật sự khác dữ liệu thật hay chỉ chép lại.
- AI4: huấn luyện mô hình IDS (Random Forest, XGBoost) trên dữ liệu thật và thật + CTGAN, rồi so sánh.
- AI5: kết hợp điểm rủi ro MIA của nhóm Cyber Security để lọc hoặc sinh lại bản ghi rủi ro cao.